In [ ]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers transformers pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
pip install transformers==4.41.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
#“How do you create a simple LangChain chain that takes a user topic and generates an answer?”

from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

pipe=pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=100
)

llm=HuggingFacePipeline(pipeline=pipe)

prompt=PromptTemplate.from_template(
    "Explain {topic} in simple interview language in 5 lines"
)

chain=prompt|llm

answer=chain.invoke({"topic":"rag pipeline"})
print(answer)

# This is a basic LCEL chain. The input first goes into a prompt template, then
# the formatted prompt is passed to the LLM. The | operator connects components, and
#  .invoke() runs the chain. LCEL is useful because the same style can later support
#  retrieval, parsing, batching, and streaming.


Describe rag pipeline in simple interview language in 5 lines


In [ ]:
#“In RAG, why do we split documents, and how would you do it in LangChain?”

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text="""
  Tower Research Capital is a quantitative trading firm.
The company works on high-performance trading infrastructure.
Engineers work on low latency systems, machine learning, data pipelines, and research tools.
A RAG system can help employees query internal documents, experiment notes, and technical runbooks.
"""

docs=[Document(page_content=text)]

splitter=RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20
)

chunks=splitter.split_documents(docs)

for i,chunk in enumerate(chunks):
  print("CHUNK",i)
  print(chunk.page_content)
  print()

# LLMs have context limits, so large documents cannot be sent fully
# every time. We split documents into smaller overlapping chunks so retrieval
#  can fetch only the most relevant pieces. chunk_overlap helps preserve contex
#  t across chunk boundaries.

CHUNK 0
Tower Research Capital is a quantitative trading firm.
The company works on high-performance trading infrastructure.

CHUNK 1
Engineers work on low latency systems, machine learning, data pipelines, and research tools.

CHUNK 2
A RAG system can help employees query internal documents, experiment notes, and technical runbooks.



In [ ]:
#“Can you build a small RAG pipeline without OpenAI or paid APIs?”

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline

raw_text="""
  Tower Research Capital is a quantitative trading firm.
It focuses on high-performance platforms and independent trading teams.
Engineers work on low-latency programming, FPGA technology, hardware acceleration, and machine learning.
The role requires building data pipelines, ML models, NLP models, experiments, and MLOps deployments.
A RAG system can help employees search internal research notes, documentation, model reports, and runbooks.
"""

docs=[Document(page_content=raw_text)]

splitter=RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

chunks=splitter.split_documents(raw_text)



In [ ]:
# “Can you build a small RAG pipeline without OpenAI or paid APIs?”

# =========================
# 1. Imports
# =========================
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline

# =========================
# 2. Raw Data
# =========================
raw_text = """
Tower Research Capital is a quantitative trading firm.
It focuses on high-performance platforms and independent trading teams.
Engineers work on low-latency programming, FPGA technology, hardware acceleration, and machine learning.
The role requires building data pipelines, ML models, NLP models, experiments, and MLOps deployments.
A RAG system can help employees search internal research notes, documentation, model reports, and runbooks.
"""

docs = [Document(page_content=raw_text)]

# =========================
# 3. Chunking
# =========================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

chunks = splitter.split_documents(docs)

# =========================
# 4. Embeddings
# =========================
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# =========================
# 5. Vector Store (FAISS)
# =========================
vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# =========================
# 6. LLM (IMPORTANT FIX HERE)
# =========================
pipe = pipeline(
    "text-generation",   # <-- FIX for your environment
    model="google/flan-t5-small",
    max_new_tokens=120
)

llm = HuggingFacePipeline(pipeline=pipe)

# =========================
# 7. Prompt Template
# =========================
prompt = PromptTemplate.from_template(
"""
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
"""
)

# =========================
# 8. Query
# =========================
question = "What kind of work do engineers do at Tower?"

retrieved_docs = retriever.invoke(question)

context = "\n\n".join(doc.page_content for doc in retrieved_docs)

# =========================
# 9. RAG Chain
# =========================
chain = prompt | llm

ans = chain.invoke({
    "context": context,
    "question": question
})

# =========================
# 10. Output
# =========================
print("Retrieved context:\n")
print(context)

print("\nFinal answer:\n")
print(ans)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCa

Retrieved context:

Engineers work on low-latency programming, FPGA technology, hardware acceleration, and machine learning.

Tower Research Capital is a quantitative trading firm.
It focuses on high-performance platforms and independent trading teams.

Final answer:


Answer the question using the context below.

Context:
Engineers work on low-latency programming, FPGA technology, hardware acceleration, and machine learning.

Tower Research Capital is a quantitative trading firm.
It focuses on high-performance platforms and independent trading teams.

Question:
What kind of work do engineers do at Tower?

Answer:



In [ ]:
#Langgraph

from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END

class State(TypedDict):
  question:str
  answer:str

def answer_node(state:State):
  question=state["question"]
  return {
      "answer": f"You  asked {question}. This is basic LangGraph response"
  }

builder=StateGraph(State)
builder.add_node("answer_node",answer_node)
builder.add_edge(START,"answer_node")
builder.add_edge("answer_node",END)
graph=builder.compile()
result=graph.invoke({
    "question":"What is RAG?"
})
print(result["answer"])



You  asked What is RAG?. This is basic LangGraph response
